# Implement Attention from Scratch
### Problem Statement
Multi-Head Attention (MHA) is the bread-and-butter of the Transformer architecture. It enables the model to **jointly attend** to information from different representation subspaces at different positions.

Your goal is to implement MHA from scratch using PyTorch, simulating exactly what `torch.nn.MultiheadAttention` does — projecting Q, K, V for each head, computing attention weights, applying them to V, and concatenating the outputs across all heads.

---

### Requirements

1. **Linear Projections for Q, K, V**
   - Project input `q`, `k`, `v` into a total of `d_model` dimensions.
   - Split them into `num_heads` of `d_head = d_model // num_heads` each.

2. **Scaled Dot-Product Attention per Head**
   - Compute attention scores:  
     `scores = Q @ Kᵀ / sqrt(d_head)`
   - Apply an optional `mask` before softmax.
   - Use the scores to weight `V`.

3. **Combine the Heads**
   - Concatenate the outputs of all heads.
   - Apply a final linear projection to restore the shape: `(batch_size, seq_len, d_model)`.

4. **Validate Against PyTorch’s Reference**
   - Test your output against `torch.nn.MultiheadAttention` using the same input tensors.
   - Check for numerical closeness using `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Make sure all tensors are reshaped properly when splitting and combining heads.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention` output when heads and shape are aligned.

---

<details>
  <summary>💡 Hint</summary>

  - Use `.view()` and `.transpose()` to shape Q, K, V to `(batch_size, num_heads, seq_len, d_head)`.
  - Softmax should be applied over the **last dimension** (attention scores across sequence).
  - Use `.contiguous().view()` to flatten the multi-head outputs back into `(batch_size, seq_len, d_model)`.
  - Match PyTorch’s behavior using the same projections and batch-first format.

</details>

---

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [26]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

torch.Size([3, 4, 8])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F




class MultiHeadAttention(nn.Module):
    """
    Implements multi-head attention.
    """
    def __init__(self, num_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads  # Head size dimension

        self.Q_w = nn.Linear(d_model, d_model, bias=False)
        self.K_w = nn.Linear(d_model, d_model, bias=False)
        self.V_w = nn.Linear(d_model, d_model, bias=False)
        self.W_out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, mask=None):
        """
        Args:
            q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
            k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
            v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Masking tensor for attention

        Returns:
            Tensor: Multi-head attention output of shape (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, _ = q.shape

        Q = self.Q_w(q)  # (batch_size, seq_len, d_model)
        K = self.K_w(k)
        V = self.V_w(v)

        # Reshape to (batch_size, num_heads, seq_len, d_head)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)

        # Mask check
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)

        output = torch.matmul(attn_weights, V)  # (batch_size, num_heads, seq_len, d_head)

        # Combine heads
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_out(output)

## 📚 Understanding the Head Combination Step

The most confusing line in multi-head attention is often:
```python
output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
```

Let's break down what each operation does:

### Starting Point
Before this line, `output` has shape: `(batch_size, num_heads, seq_len, d_head)`

For example: `(3, 2, 4, 4)` means:
- 3 samples in the batch
- 2 attention heads  
- 4 tokens in sequence
- 4 dimensions per head (where `d_model=8`, so `d_head=8/2=4`)

---

### Step 1: `.transpose(1, 2)` — Swap dimensions

Swaps dimensions 1 and 2 (num_heads ↔ seq_len):

```
Before: (batch_size, num_heads, seq_len, d_head)  →  (3, 2, 4, 4)
After:  (batch_size, seq_len, num_heads, d_head)  →  (3, 4, 2, 4)
```

**Why?** We need `seq_len` and `num_heads` next to each other so we can merge them.

---

### Step 2: `.contiguous()` — Fix memory layout

Makes the tensor's memory layout contiguous in RAM.

**Why?** `.transpose()` doesn't actually move data in memory—it just changes how PyTorch *views* it. Before we can use `.view()` to reshape, the memory must be laid out sequentially. `.contiguous()` reorganizes the data if needed.

---

### Step 3: `.view(batch_size, seq_len, d_model)` — Merge heads

Reshapes by concatenating the last two dimensions:

```
Before: (batch_size, seq_len, num_heads, d_head)  →  (3, 4, 2, 4)
After:  (batch_size, seq_len, d_model)            →  (3, 4, 8)
```

**What happens?** `num_heads × d_head = 2 × 4 = 8 = d_model` ✅

All heads are **concatenated** together for each token!

---

### Visual Example

```python
# Multi-head output: (batch=1, heads=2, seq=3, d_head=4)
[[[
    [[a1, a2, a3, a4],    # Head 0, Token 0
     [b1, b2, b3, b4],    # Head 0, Token 1  
     [c1, c2, c3, c4]],   # Head 0, Token 2
    
    [[d1, d2, d3, d4],    # Head 1, Token 0
     [e1, e2, e3, e4],    # Head 1, Token 1
     [f1, f2, f3, f4]]    # Head 1, Token 2
]]]

# After transpose(1,2): (batch=1, seq=3, heads=2, d_head=4)
[[[
    [[a1, a2, a3, a4], [d1, d2, d3, d4]],  # Token 0: [Head0, Head1]
    [[b1, b2, b3, b4], [e1, e2, e3, e4]],  # Token 1: [Head0, Head1]
    [[c1, c2, c3, c4], [f1, f2, f3, f4]]   # Token 2: [Head0, Head1]
]]]

# After view(1, 3, 8): (batch=1, seq=3, d_model=8)
[[[
    [a1, a2, a3, a4, d1, d2, d3, d4],  # Token 0: heads concatenated!
    [b1, b2, b3, b4, e1, e2, e3, e4],  # Token 1: heads concatenated!
    [c1, c2, c3, c4, f1, f2, f3, f4]   # Token 2: heads concatenated!
]]]
```

---

### Summary Table

| Operation | Input Shape | Output Shape | Purpose |
|-----------|-------------|--------------|---------|
| `.transpose(1,2)` | `(B, H, S, D)` | `(B, S, H, D)` | Move seq_len next to num_heads |
| `.contiguous()` | `(B, S, H, D)` | `(B, S, H, D)` | Fix memory layout for `.view()` |
| `.view(B, S, H×D)` | `(B, S, H, D)` | `(B, S, d_model)` | Concatenate all heads |

Where: `B=batch`, `H=num_heads`, `S=seq_len`, `D=d_head`, `H×D=d_model`

In [ ]:
# Testing on data & compare
# Create PyTorch's MultiheadAttention
multihead_attn = torch.nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, bias=False, batch_first=True)

# Create our custom implementation
custom_attn = MultiHeadAttention(num_heads, d_model)

# Copy weights from PyTorch's implementation to ours for fair comparison
custom_attn.Q_w.weight.data = multihead_attn.in_proj_weight[:d_model, :].clone()
custom_attn.K_w.weight.data = multihead_attn.in_proj_weight[d_model:2*d_model, :].clone()
custom_attn.V_w.weight.data = multihead_attn.in_proj_weight[2*d_model:, :].clone()
custom_attn.W_out.weight.data = multihead_attn.out_proj.weight.data.clone()

# Now compare outputs
output_custom = custom_attn(q, k, v)
print("Custom output:")
print(output_custom)

output, _ = multihead_attn(q, k, v)
print("\nPyTorch output:")
print(output)

assert torch.allclose(output_custom, output, atol=1e-06, rtol=1e-05)  # Check if they are close enough.
print("\n✅ Outputs match! Implementation is correct.")